# 02 — Recorte, rótulos de referência e protocolo (Etapa 2)

Gera, para cada AOI, a **grade de patches**, a **divisão espacial** e os **rótulos de avaliação**
(o modelo nunca os vê). O protocolo fixado está em [PROTOCOLO.md](PROTOCOLO.md).

| AOI | Papel | Caixa (lon/lat) | UTM |
|---|---|---|---|
| `tapajos` | principal (treino/val/teste) | [-57, -7,5, -55, -5,5] | 21S |
| `norte_ro` | generalização (só teste) | [-64, -10, -62, -8] | 20S |

| Fonte | Como entra | Resolução temporal |
|---|---|---|
| PRODES (INPE) | polígonos via WFS TerraBrasilis | anual (data da imagem) |
| DETER (INPE) | polígonos via WFS TerraBrasilis | data do alerta |
| Hansen GFC | `lossyear`, Earth Engine | anual |
| MapBiomas LULC | transições Y-1 → Y, Earth Engine | anual |
| MODIS MCD64A1 | área queimada sobre floresta, Earth Engine | trimestral |

**Onde rodar:** Colab (Earth Engine autenticado). Localmente, sem credenciais do Earth Engine, as fontes raster são puladas e o resumo registra isso.
Saídas grandes vão para `DATA_ROOT/rotulos/<aoi>/`; o resumo e as figuras vão para `results/etapa2_rotulos/<timestamp>/`.

## 1. Setup

In [ ]:
import sys, os, subprocess, time
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'github.com/GabrielPow/ProjetoGrassQuality.git'
BRANCH = 'main'
REPO_DIR = '/content/ProjetoGrassQuality'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = None
    url = f'https://{token}@{REPO_URL}' if token else f'https://{REPO_URL}'
    if os.path.exists(REPO_DIR):
        subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)
    else:
        subprocess.run(['git', 'clone', '-b', BRANCH, url, REPO_DIR], check=True)

    from importlib import metadata
    import numpy
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    f'{REPO_DIR}/notebooks/gabriel/requirements-colab.txt'], check=True)
    if metadata.version('numpy') != numpy.__version__:
        print(f'numpy {numpy.__version__} -> {metadata.version("numpy")}: reiniciando o ambiente. '
              'Rode todas as células de novo (a instalação não se repete).')
        os.kill(os.getpid(), 9)
    os.chdir(f'{REPO_DIR}/notebooks/gabriel')
else:
    print('Local: setup do Colab pulado.')

In [ ]:
import json
import numpy as np
import pandas as pd
import config, utils, rotulos, avaliacao

config.make_dirs()
utils.set_seed(config.SEED)
run_dir = utils.new_run('etapa2_rotulos')
print('rótulos em', config.LABELS_DIR, '| resumo em', run_dir)

USE_EE = True
try:
    import ee
    try:
        ee.Initialize(project=config.EE_PROJECT_ID)
    except Exception:
        if not IN_COLAB:
            raise
        ee.Authenticate()
        ee.Initialize(project=config.EE_PROJECT_ID)
except Exception as e:
    USE_EE = False
    print('Earth Engine indisponível, fontes raster puladas:', str(e)[:120])
print('Earth Engine:', USE_EE)

## 2. Grade de patches e divisão espacial

Patches de 224 px × 10 m = **2240 m**, alinhados a múltiplos de 2240 m em UTM (a Etapa 3 usa a mesma grade).
Blocos de 4 × 4 patches (8,96 km) vão inteiros para uma partição: train/val/test 70/15/15 na AOI principal; tudo `test_gen` na AOI de generalização.

In [ ]:
grids = {}
for aoi, a in config.AOIS.items():
    g = rotulos.make_grid(aoi)
    g['split'] = rotulos.assign_blocks_to_splits(g) if a['role'] == 'principal' else 'test_gen'
    assert (g.groupby('block')['split'].nunique() == 1).all(), 'vazamento espacial: bloco em mais de uma partição'
    grids[aoi] = g
    print(f"{aoi:<9} {len(g):>6} patches  {g['block'].nunique():>4} blocos  ",
          g['split'].value_counts(normalize=True).round(3).to_dict())

## 3. Fontes vetoriais: PRODES e DETER (WFS TerraBrasilis)

In [ ]:
vec, polys = {}, {}
for aoi, g in grids.items():
    cache = config.LABELS_DIR / aoi / 'cache'
    t0 = time.time()
    prodes, deter = rotulos.fetch_prodes(aoi, cache), rotulos.fetch_deter(aoi, cache)
    vec[aoi] = pd.concat([rotulos.overlay_events(g, prodes), rotulos.overlay_events(g, deter)], ignore_index=True)
    polys[aoi] = pd.concat([rotulos.event_table(prodes), rotulos.event_table(deter)], ignore_index=True)
    print(f"{aoi}: PRODES {len(prodes)}, DETER {len(deter)} {deter['tipo'].value_counts().to_dict()} "
          f"-> {len(vec[aoi])} pares evento-patch ({time.time() - t0:.0f}s)")

## 4. Fontes raster: Hansen, MapBiomas, MCD64A1 (Earth Engine)

Média por patch de imagens binárias = fração do patch afetada. Em lotes com cache por lote: se cair, rode de novo e ele continua de onde parou.
- **MapBiomas:** desmatamento = floresta natural em Y-1 → uso antrópico em Y; mineração = entrada na classe 30; água = ganho ou perda da classe 33. Também a cobertura do ano-base (2019) para estratificar.
- **MCD64A1:** queimada por trimestre, só sobre floresta natural (MapBiomas do ano anterior). Fogo em pastagem é manejo, não anomalia.

In [ ]:
ras, meta = {}, {}
if USE_EE:
    sources = {'hansen': (rotulos.hansen_image, 30), 'mapbiomas': (rotulos.mapbiomas_image, 30),
               'mcd64': (rotulos.mcd64_image, 100)}
    for aoi, g in grids.items():
        cache = config.LABELS_DIR / aoi / 'cache'
        ras[aoi], meta[aoi] = {}, {}
        for name, (make, scale) in sources.items():
            t0 = time.time()
            img, info = make()
            ras[aoi][name] = rotulos.ee_reduce(img, g, name, cache, scale=scale)
            meta[aoi][name] = info
            print(f'{aoi} {name}: {info} ({time.time() - t0:.0f}s)')
else:
    print('Pulado (sem Earth Engine).')

## 5. Montagem: eventos, cobertura, normais e rótulos anual / trimestral

In [ ]:
out = {}
for aoi, g in grids.items():
    parts = [vec[aoi]] + ([rotulos.raster_events(w) for w in ras[aoi].values()] if USE_EE else [])
    eventos = pd.concat(parts, ignore_index=True)
    pid = g['patch_id'].values

    grade = pd.DataFrame(g.drop(columns='geometry'))
    if USE_EE:
        cover = ras[aoi]['mapbiomas'].set_index('patch_id').filter(like='cover__')
        cover.columns = cover.columns.str.replace('cover__', 'cob_')
        grade = grade.merge(cover, left_on='patch_id', right_index=True, how='left')
        grade['cobertura'] = cover.idxmax(axis=1).str.replace('cob_', '').reindex(grade['patch_id']).values
    else:
        grade['cobertura'] = 'desconhecida'
    grade = grade.merge(rotulos.normal_flags(eventos, pid), on='patch_id')

    anual, trim = rotulos.annual_labels(eventos, pid), rotulos.quarterly_labels(eventos, pid)

    d = config.LABELS_DIR / aoi
    g[['patch_id', 'split', 'geometry']].to_file(d / 'grade.gpkg', driver='GPKG')
    grade.to_csv(d / 'grade.csv', index=False)
    eventos.to_csv(d / 'eventos.csv.gz', index=False)
    polys[aoi].to_csv(d / 'poligonos.csv.gz', index=False)
    anual.to_csv(d / 'rotulos_anuais.csv.gz', index=False)
    trim.to_csv(d / 'rotulos_trimestrais.csv.gz', index=False)
    out[aoi] = dict(grade=grade, eventos=eventos, anual=anual, trim=trim)
    print(f'{aoi}: salvo em {d}')

## 6. Verificações e resumo

O que olhar:
1. **Prevalência real** por tipo, partição e período (anual e trimestral). É contra essa prevalência que as métricas serão reportadas.
2. **Normais de treino** por cobertura: o modelo precisa de floresta, pastagem, agricultura **e** água estáveis. Se algum grupo sumir, rever `NORMAL_MAX_FRAC`.
3. **Concordância entre fontes** de desmatamento (PRODES × Hansen × MapBiomas): mede quão independentes os rótulos são.

In [ ]:
def prevalence(lab, grade, key):
    df = lab.merge(grade[['patch_id', 'split']], on='patch_id')
    rows = []
    for (split, per), x in df.groupby(['split', 'periodo']):
        r = {'split': split, 'periodo': per, 'n': len(x), 'ambiguo': (x['estado'] == 'ambiguo').mean(),
             'qualquer': (x['estado'] == 'evento').mean()}
        r.update({t: (x[f'frac_{t}'] >= config.EVENT_MIN_FRAC).mean() for t in config.EVENT_TYPES})
        rows.append(r)
    return pd.DataFrame(rows)

resumo = {'use_ee': USE_EE, 'fontes_raster': meta, 'config': {k: getattr(config, k) for k in
          ['EVENT_MIN_FRAC', 'NORMAL_MAX_FRAC', 'PATCH_M', 'BLOCK_PATCHES', 'SPLIT_TARGET', 'TRAIN_END_YEAR']}}
for aoi, o in out.items():
    print(f'\n===== {aoi}')
    pa, pq = prevalence(o['anual'], o['grade'], 'year'), prevalence(o['trim'], o['grade'], 'step')
    print('Prevalência por patch-ano (todas as fontes):'); display(pa.round(4))
    print('Prevalência por patch-trimestre (DETER + MCD64):'); display(pq.round(4))
    nt = o['grade'].groupby(['cobertura', 'split'])['normal_treino'].agg(['sum', 'mean']).unstack('split')
    print('Normais de treino por cobertura (n e fração do grupo):'); display(nt.round(3))
    resumo[aoi] = {'n_patches': len(o['grade']), 'n_blocos': int(o['grade']['block'].nunique()),
                   'splits': o['grade']['split'].value_counts().to_dict(),
                   'normais_treino': int(o['grade']['normal_treino'].sum()),
                   'normais_estritos': int(o['grade']['normal_estrito'].sum()),
                   'prevalencia_anual': pa.to_dict('records'), 'prevalencia_trimestral': pq.to_dict('records')}
    pa.to_csv(run_dir / f'prevalencia_anual_{aoi}.csv', index=False)
    pq.to_csv(run_dir / f'prevalencia_trimestral_{aoi}.csv', index=False)
    nt.to_csv(run_dir / f'normais_por_cobertura_{aoi}.csv')

In [ ]:
# concordância de desmatamento entre fontes, por patch-ano (>= EVENT_MIN_FRAC)
for aoi, o in out.items():
    ev = o['eventos'][o['eventos']['tipo'] == 'desmatamento']
    ev = ev.assign(year=pd.to_datetime(ev['date_end']).dt.year)
    s = ev.groupby(['patch_id', 'year', 'source'])['frac'].sum().unstack('source').fillna(0) >= config.EVENT_MIN_FRAC
    s = s[s.any(axis=1)]
    srcs = list(s.columns)
    agree = pd.DataFrame({a: {b: (s[a] & s[b]).sum() / max(s[a].sum(), 1) for b in srcs} for a in srcs}).T
    print(f'{aoi}: P(coluna positiva | linha positiva), patch-anos com desmatamento')
    display(agree.round(2))
    resumo[aoi]['concordancia_desmatamento'] = agree.round(3).to_dict()

## 7. Mapas: partições e normais de treino

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e1e0d9'
SPLIT_COLORS = {'train': '#2a78d6', 'val': '#eb6834', 'test': '#1baf7a', 'test_gen': '#eda100'}
plt.rcParams.update({'font.size': 9, 'axes.edgecolor': GRID, 'axes.labelcolor': MUTED,
                     'xtick.color': MUTED, 'ytick.color': MUTED})

fig, axes = plt.subplots(len(grids), 2, figsize=(11, 5.2 * len(grids)), squeeze=False)
for (aoi, g), row in zip(grids.items(), axes):
    grade = out[aoi]['grade']
    gg = g.merge(grade[['patch_id', 'normal_treino']], on='patch_id')
    gg.plot(ax=row[0], color=gg['split'].map(SPLIT_COLORS), edgecolor='none')
    below = dict(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.01), ncol=4)
    row[0].legend(handles=[Patch(color=c, label=s) for s, c in SPLIT_COLORS.items() if s in set(gg['split'])],
                  **below)
    row[0].set_title(f'{aoi}: partições (blocos de {config.BLOCK_PATCHES * config.PATCH_M / 1000:.2f} km)', color=INK)
    gg.plot(ax=row[1], color=np.where(gg['normal_treino'], '#2a78d6', '#e1e0d9'), edgecolor='none')
    row[1].legend(handles=[Patch(color='#2a78d6', label=f"normal ({gg['normal_treino'].mean():.0%})"),
                           Patch(color='#e1e0d9', label='com evento em 2019–2025')], **below)
    row[1].set_title(f'{aoi}: normais de treino', color=INK)
    for ax in row:
        ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel(''); ax.set_ylabel('')
fig.tight_layout()
fig.savefig(run_dir / 'mapa_particoes_normais.png', dpi=130)
plt.show()

## 8. Teste do protocolo de avaliação com scores sintéticos

Não é resultado: só confirma que `avaliacao.py` roda de ponta a ponta sobre os rótulos reais.
Score = ruído + sinal nos patch-trimestres com evento. Espera-se AUROC alto e AUPRC limitado pela prevalência.

In [ ]:
aoi = 'tapajos'
q, grade = out[aoi]['trim'], out[aoi]['grade']
rng = np.random.default_rng(config.SEED)
sc = q[['patch_id', 'step']].assign(
    score=rng.normal(size=len(q)) + 2 * (q.filter(like='frac_').max(axis=1) >= config.EVENT_MIN_FRAC).values)

display(avaliacao.evaluate(sc, q, grade, n_boot=100).round(3))
thr = avaliacao.threshold_at_fpr(sc, q, grade, fpr=0.01)
pol = polys[aoi]
display(avaliacao.event_recall(sc, out[aoi]['eventos'], pol, thr).round(3))
lt = avaliacao.lead_time(sc, out[aoi]['eventos'], q, thr)
print('lead time (trimestres):', lt['lag_trimestres'].value_counts().sort_index().to_dict(),
      '| taxa ao acaso:', round(avaliacao.chance_rate(sc, q, thr), 4))

## 9. Salvar o resumo

In [ ]:
(run_dir / 'resumo.json').write_text(json.dumps(resumo, indent=2, default=str), encoding='utf-8')
print('Salvo em', run_dir)
print(sorted(p.name for p in run_dir.iterdir()))